In [1]:
%cd /content
!rm -rf /content/void-model
!git clone https://github.com/Netflix/void-model.git
!ls /content/void-model

/content
Cloning into 'void-model'...
remote: Enumerating objects: 297, done.
remote: Counting objects: 100% (109/109), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 297 (delta 77), reused 83 (delta 59), pack-reused 188 (from 2)
Receiving objects: 100% (297/297), 35.00 MiB | 31.36 MiB/s, done.
Resolving deltas: 100% (107/107), done.
assets		 datasets   notebook.ipynb    sample	  VLM-MASK-REASONER
config		 inference  README.md	      scripts
data_generation  LICENSE    requirements.txt  videox_fun


In [2]:
%cd /content/void-model
!python -m pip install -q --upgrade pip
!python -m pip install -q -U huggingface_hub
!python -m pip install -q diffusers accelerate transformers safetensors peft \
    opencv-python scikit-image imageio imageio-ffmpeg mediapy decord kornia \
    albumentations timm tomesd Pillow numpy scipy scikit-learn datasets einops \
    omegaconf ml_collections absl-py loguru tqdm matplotlib sentencepiece ftfy \
    beautifulsoup4 func-timeout requests packaging lpips torchdiffeq torchsde

/content/void-model
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 43.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
!mkdir -p /content/models

from huggingface_hub import snapshot_download, hf_hub_download

snapshot_download(
    repo_id="alibaba-pai/CogVideoX-Fun-V1.5-5b-InP",
    local_dir="/content/models/CogVideoX-Fun-V1.5-5b-InP",
    local_dir_use_symlinks=False,
)

hf_hub_download(
    repo_id="netflix/void-model",
    filename="void_pass1.safetensors",
    local_dir="/content/models",
    local_dir_use_symlinks=False,
)

hf_hub_download(
    repo_id="netflix/void-model",
    filename="void_pass2.safetensors",
    local_dir="/content/models",
    local_dir_use_symlinks=False,
)

print("Downloads concluídos.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


void_pass1.safetensors:   0%|          | 0.00/11.1G [00:00<?, ?B/s]

void_pass2.safetensors:   0%|          | 0.00/11.1G [00:00<?, ?B/s]

Downloads concluídos.


In [4]:
from pathlib import Path

repo = Path('/content/void-model')

def replace_once(path: Path, old: str, new: str):
    text = path.read_text(encoding='utf-8')
    if new in text:
        return
    if old not in text:
        raise ValueError(f'Trecho não encontrado em {path}')
    path.write_text(text.replace(old, new), encoding='utf-8')

temporal_metrics_py = r'''import csv
import json
import math
import os
from typing import Any, Dict, List, Optional

import numpy as np
import torch
import torch.nn.functional as F
from loguru import logger
from skimage.metrics import structural_similarity

from videox_fun.utils.optical_flow_utils import RAFTFlowExtractor


def normalize_video_for_metrics(video: torch.Tensor, valid_frames: Optional[int] = None) -> torch.Tensor:
    if video.ndim == 5:
        video = video[0]
    if video.ndim != 4:
        raise ValueError(f"Expected [B,C,T,H,W] or [C,T,H,W], got shape {tuple(video.shape)}")

    if video.shape[0] != 3 and video.shape[1] == 3:
        video = video.permute(1, 0, 2, 3)
    if video.shape[0] != 3:
        raise ValueError(f"Expected RGB channels, got shape {tuple(video.shape)}")

    frames = video.detach().float().permute(1, 0, 2, 3).contiguous()
    if valid_frames is not None:
        frames = frames[:valid_frames]

    if frames.numel() == 0:
        raise ValueError("Video has no frames after normalization")

    if frames.max() > 1.5:
        frames = frames / 255.0
    elif frames.min() < 0.0:
        frames = (frames + 1.0) / 2.0

    return frames.clamp(0.0, 1.0)


def _compute_psnr(frame1: np.ndarray, frame2: np.ndarray) -> float:
    mse = float(np.mean((frame1 - frame2) ** 2))
    mse = max(mse, 1e-10)
    return float(10.0 * math.log10(1.0 / mse))


def _normalized_base_grid(height: int, width: int, device: torch.device, dtype: torch.dtype) -> torch.Tensor:
    y_coords = torch.linspace(-1.0, 1.0, steps=height, device=device, dtype=dtype)
    x_coords = torch.linspace(-1.0, 1.0, steps=width, device=device, dtype=dtype)
    grid_y, grid_x = torch.meshgrid(y_coords, x_coords, indexing="ij")
    return torch.stack((grid_x, grid_y), dim=-1).unsqueeze(0)


def backward_warp(frame: torch.Tensor, backward_flow: torch.Tensor) -> torch.Tensor:
    _, _, height, width = frame.shape
    base_grid = _normalized_base_grid(height, width, frame.device, frame.dtype)

    flow_x = backward_flow[:, 0] * (2.0 / max(width - 1, 1))
    flow_y = backward_flow[:, 1] * (2.0 / max(height - 1, 1))
    flow_grid = torch.stack((flow_x, flow_y), dim=-1)
    sample_grid = base_grid + flow_grid

    return F.grid_sample(
        frame,
        sample_grid,
        mode="bilinear",
        padding_mode="zeros",
        align_corners=True,
    )


def summarize_pair_metrics(pair_metrics: List[Dict[str, float]]) -> Dict[str, float]:
    metric_names = [
        "lpips_temporal",
        "optical_flow_consistency_l1",
        "psnr_consecutive",
        "ssim_consecutive",
    ]
    summary: Dict[str, float] = {}

    for metric_name in metric_names:
        values = [float(pair[metric_name]) for pair in pair_metrics]
        if not values:
            continue
        array = np.asarray(values, dtype=np.float64)
        summary[f"{metric_name}_mean"] = float(array.mean())
        summary[f"{metric_name}_std"] = float(array.std())
        summary[f"{metric_name}_min"] = float(array.min())
        summary[f"{metric_name}_max"] = float(array.max())

    summary["num_pairs"] = len(pair_metrics)
    return summary


class TemporalMetricsEvaluator:
    def __init__(
        self,
        device: Optional[str] = None,
        lpips_model: Optional[torch.nn.Module] = None,
        flow_extractor: Optional[RAFTFlowExtractor] = None,
    ) -> None:
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self._lpips_model = lpips_model
        self._flow_extractor = flow_extractor

    def _get_lpips_model(self) -> torch.nn.Module:
        if self._lpips_model is None:
            import lpips
            self._lpips_model = lpips.LPIPS(net="alex").to(self.device)
            self._lpips_model.eval()
        return self._lpips_model

    def _get_flow_extractor(self) -> RAFTFlowExtractor:
        if self._flow_extractor is None:
            self._flow_extractor = RAFTFlowExtractor(device=self.device)
        return self._flow_extractor

    def _compute_pair_metrics(self, frame1: torch.Tensor, frame2: torch.Tensor) -> Dict[str, float]:
        frame1_cpu = frame1.detach().cpu()
        frame2_cpu = frame2.detach().cpu()
        frame1_np = frame1_cpu.permute(1, 2, 0).numpy()
        frame2_np = frame2_cpu.permute(1, 2, 0).numpy()

        psnr = _compute_psnr(frame1_np, frame2_np)

        min_hw = min(frame1_np.shape[0], frame1_np.shape[1])
        win_size = min(7, min_hw)
        if win_size % 2 == 0:
            win_size -= 1
        if win_size < 3:
            win_size = 3

        ssim = float(
            structural_similarity(
                frame1_np,
                frame2_np,
                channel_axis=2,
                data_range=1.0,
                win_size=win_size,
            )
        )

        lpips_model = self._get_lpips_model()
        lpips_input_1 = frame1.unsqueeze(0).to(self.device) * 2.0 - 1.0
        lpips_input_2 = frame2.unsqueeze(0).to(self.device) * 2.0 - 1.0
        with torch.no_grad():
            lpips_value = float(lpips_model(lpips_input_1, lpips_input_2).mean().item())

        flow_extractor = self._get_flow_extractor()
        flow_frame1 = frame1.unsqueeze(0).to(self.device)
        flow_frame2 = frame2.unsqueeze(0).to(self.device)
        with torch.no_grad():
            backward_flow = flow_extractor.extract_flow(flow_frame2, flow_frame1)
            warped_frame1 = backward_warp(flow_frame1, backward_flow)
            valid_mask = backward_warp(
                torch.ones((1, 1, frame1.shape[1], frame1.shape[2]), device=self.device, dtype=flow_frame1.dtype),
                backward_flow,
            )

        diff = torch.abs(warped_frame1 - flow_frame2)
        valid_pixels = valid_mask > 0.5
        if valid_pixels.any():
            flow_consistency = float(diff.masked_select(valid_pixels.expand_as(diff)).mean().item())
        else:
            flow_consistency = float(diff.mean().item())

        return {
            "lpips_temporal": lpips_value,
            "optical_flow_consistency_l1": flow_consistency,
            "psnr_consecutive": psnr,
            "ssim_consecutive": ssim,
        }

    def evaluate(
        self,
        video: torch.Tensor,
        *,
        video_name: Optional[str] = None,
        stage: Optional[str] = None,
        fps: Optional[int] = None,
        valid_frames: Optional[int] = None,
    ) -> Dict[str, Any]:
        frames = normalize_video_for_metrics(video, valid_frames=valid_frames)
        if frames.shape[0] < 2:
            raise ValueError("Temporal metrics require at least two frames")

        pair_metrics: List[Dict[str, float]] = []
        for index in range(frames.shape[0] - 1):
            metrics = self._compute_pair_metrics(frames[index], frames[index + 1])
            pair_metrics.append({
                "t": index,
                "t_next": index + 1,
                **metrics,
            })

        return {
            "video_name": video_name,
            "stage": stage,
            "num_frames": int(frames.shape[0]),
            "num_pairs": int(frames.shape[0] - 1),
            "fps": fps,
            "summary": summarize_pair_metrics(pair_metrics),
            "pairs": pair_metrics,
        }

    @staticmethod
    def format_summary(result: Dict[str, Any]) -> str:
        summary = result["summary"]
        video_name = result.get("video_name") or "video"
        stage = result.get("stage") or "temporal_eval"
        return (
            f"[temporal_eval] {video_name} {stage} | pairs={summary['num_pairs']} | "
            f"lpips={summary['lpips_temporal_mean']:.4f} | "
            f"ofc_l1={summary['optical_flow_consistency_l1_mean']:.4f} | "
            f"psnr={summary['psnr_consecutive_mean']:.2f} | "
            f"ssim={summary['ssim_consecutive_mean']:.4f}"
        )

    @staticmethod
    def write(result: Dict[str, Any], output_prefix: str) -> Dict[str, str]:
        json_path = f"{output_prefix}_temporal_metrics.json"
        csv_path = f"{output_prefix}_temporal_metrics_pairs.csv"
        os.makedirs(os.path.dirname(json_path), exist_ok=True)

        with open(json_path, "w", encoding="utf-8") as json_file:
            json.dump(result, json_file, indent=2)

        with open(csv_path, "w", newline="", encoding="utf-8") as csv_file:
            fieldnames = [
                "video_name",
                "stage",
                "t",
                "t_next",
                "lpips_temporal",
                "optical_flow_consistency_l1",
                "psnr_consecutive",
                "ssim_consecutive",
            ]
            writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
            writer.writeheader()
            for pair in result["pairs"]:
                writer.writerow({
                    "video_name": result.get("video_name"),
                    "stage": result.get("stage"),
                    **pair,
                })

        logger.info(TemporalMetricsEvaluator.format_summary(result))
        return {"json": json_path, "csv": csv_path}
'''

test_temporal_metrics_py = r'''import csv
import json
import tempfile
import unittest
import os
import sys

sys.path.insert(0, "/content/void-model")

import torch
from videox_fun.utils.temporal_metrics import TemporalMetricsEvaluator, normalize_video_for_metrics


class DummyLPIPS(torch.nn.Module):
    def forward(self, frame1, frame2):
        return torch.abs(frame1 - frame2).mean(dim=(1, 2, 3), keepdim=True)


class DummyFlowExtractor:
    def extract_flow(self, frame1, frame2):
        batch_size, _, height, width = frame1.shape
        return torch.zeros((batch_size, 2, height, width), dtype=frame1.dtype, device=frame1.device)


class TemporalMetricsTests(unittest.TestCase):
    def test_normalize_video_from_minus_one_to_one(self):
        video = torch.tensor(
            [[[[[-1.0, 1.0], [0.0, 0.5]], [[-1.0, 1.0], [0.0, 0.5]]]]],
            dtype=torch.float32,
        ).repeat(1, 3, 2, 1, 1)
        frames = normalize_video_for_metrics(video, valid_frames=1)
        self.assertEqual(frames.shape, (1, 3, 2, 2))
        self.assertGreaterEqual(float(frames.min()), 0.0)
        self.assertLessEqual(float(frames.max()), 1.0)

    def test_evaluate_and_write_sidecars(self):
        evaluator = TemporalMetricsEvaluator(
            device="cpu",
            lpips_model=DummyLPIPS(),
            flow_extractor=DummyFlowExtractor(),
        )

        video = torch.zeros((1, 3, 3, 8, 8), dtype=torch.float32)
        video[:, :, 1] = 0.25
        video[:, :, 2] = 0.5

        result = evaluator.evaluate(video, video_name="demo", stage="pass1", fps=12)
        self.assertEqual(result["num_frames"], 3)
        self.assertEqual(result["num_pairs"], 2)
        self.assertEqual(len(result["pairs"]), 2)
        self.assertIn("lpips_temporal_mean", result["summary"])

        with tempfile.TemporaryDirectory() as temp_dir:
            paths = evaluator.write(result, f"{temp_dir}/demo")
            with open(paths["json"], "r", encoding="utf-8") as json_file:
                written_json = json.load(json_file)
            self.assertEqual(written_json["video_name"], "demo")

            with open(paths["csv"], "r", encoding="utf-8") as csv_file:
                rows = list(csv.DictReader(csv_file))
            self.assertEqual(len(rows), 2)
            self.assertEqual(rows[0]["stage"], "pass1")


if __name__ == "__main__":
    unittest.main()
'''

(repo / 'videox_fun' / 'utils' / 'temporal_metrics.py').write_text(temporal_metrics_py, encoding='utf-8')
(repo / 'tests').mkdir(exist_ok=True)
(repo / 'tests' / 'test_temporal_metrics.py').write_text(test_temporal_metrics_py, encoding='utf-8')

requirements_path = repo / 'requirements.txt'
requirements_text = requirements_path.read_text(encoding='utf-8')
if 'lpips==0.1.4' not in requirements_text:
    requirements_text += '\nlpips==0.1.4\n'
    requirements_path.write_text(requirements_text, encoding='utf-8')

config_path = repo / 'config' / 'quadmask_cogvideox.py'
replace_once(
    config_path,
    "    config.skip_if_exists = True\n    config.validation = False",
    "    config.skip_if_exists = True\n    config.temporal_eval = False\n    config.validation = False",
)

predict_path = repo / 'inference' / 'cogvideox_fun' / 'predict_v2v.py'
replace_once(
    predict_path,
    "from videox_fun.utils.fp8_optimization import convert_weight_dtype_wrapper\nfrom videox_fun.utils.utils import get_video_mask_input, save_videos_grid, save_inout_row, get_video_mask_validation",
    "from videox_fun.utils.fp8_optimization import convert_weight_dtype_wrapper\nfrom videox_fun.utils.temporal_metrics import TemporalMetricsEvaluator\nfrom videox_fun.utils.utils import get_video_mask_input, save_videos_grid, save_inout_row, get_video_mask_validation",
)
replace_once(
    predict_path,
    "def run_inference(config, pipeline, vae, generator, input_video_name, keep_fg_ids=[-1]):",
    "def run_inference(config, pipeline, vae, generator, input_video_name, keep_fg_ids=[-1], temporal_metrics_evaluator=None):",
)
replace_once(
    predict_path,
    "        ).videos\n\n    if not os.path.exists(config.experiment.save_path):",
    "        ).videos\n\n    temporal_metrics_result = None\n    temporal_eval_enabled = getattr(config.experiment, \"temporal_eval\", False)\n    if temporal_eval_enabled and temporal_metrics_evaluator is not None and sample.shape[2] > 1:\n        temporal_metrics_result = temporal_metrics_evaluator.evaluate(\n            sample,\n            video_name=save_video_name,\n            stage=\"pass1\",\n            fps=config.data.fps,\n        )\n\n    if not os.path.exists(config.experiment.save_path):",
)
replace_once(
    predict_path,
    "        save_videos_grid(sample, video_path, fps=config.data.fps)\n        save_inout_row(input_video, input_video_mask, sample, video_path[:-4] + \"_tuple.mp4\", fps=config.data.fps)",
    "        save_videos_grid(sample, video_path, fps=config.data.fps)\n        save_inout_row(input_video, input_video_mask, sample, video_path[:-4] + \"_tuple.mp4\", fps=config.data.fps)\n        if temporal_metrics_result is not None:\n            temporal_metrics_evaluator.write(temporal_metrics_result, video_path[:-4])",
)
replace_once(
    predict_path,
    "    pipeline, vae, generator = load_pipeline(config)\n\n    for seq_name, fg_id in seq_fg_to_run:",
    "    pipeline, vae, generator = load_pipeline(config)\n    temporal_metrics_evaluator = None\n    if getattr(config.experiment, \"temporal_eval\", False):\n        temporal_metrics_evaluator = TemporalMetricsEvaluator()\n\n    for seq_name, fg_id in seq_fg_to_run:",
)
replace_once(
    predict_path,
    "                input_video_name=seq_name,\n                keep_fg_ids=fg_id,\n            )",
    "                input_video_name=seq_name,\n                keep_fg_ids=fg_id,\n                temporal_metrics_evaluator=temporal_metrics_evaluator,\n            )",
)

pass2_path = repo / 'inference' / 'cogvideox_fun' / 'inference_with_pass1_warped_noise.py'
replace_once(
    pass2_path,
    "from videox_fun.pipeline import CogVideoXFunInpaintPipeline\nfrom videox_fun.utils.utils import get_video_mask_input",
    "from videox_fun.pipeline import CogVideoXFunInpaintPipeline\nfrom videox_fun.utils.temporal_metrics import TemporalMetricsEvaluator\nfrom videox_fun.utils.utils import get_video_mask_input",
)
replace_once(
    pass2_path,
    "    parser.add_argument(\"--skip_noise_generation\", action=\"store_true\",\n                       help=\"Skip warped noise generation (use existing cache)\")",
    "    parser.add_argument(\"--skip_noise_generation\", action=\"store_true\",\n                       help=\"Skip warped noise generation (use existing cache)\")\n    parser.add_argument(\"--temporal_eval\", action=\"store_true\",\n                       help=\"Compute temporal evaluation sidecars for generated videos\")",
)
replace_once(
    pass2_path,
    "    os.makedirs(args.output_dir, exist_ok=True)\n",
    "    os.makedirs(args.output_dir, exist_ok=True)\n    temporal_metrics_evaluator = TemporalMetricsEvaluator() if args.temporal_eval else None\n",
)
replace_once(
    pass2_path,
    "            logger.info(f\"    Generated: {output.shape}\")\n\n            # Save results",
    "            logger.info(f\"    Generated: {output.shape}\")\n            temporal_metrics_result = None\n            if temporal_metrics_evaluator is not None and output.shape[2] > 1:\n                temporal_metrics_result = temporal_metrics_evaluator.evaluate(\n                    output,\n                    video_name=video_name,\n                    stage=\"pass2\",\n                    fps=12,\n                )\n\n            # Save results",
)
replace_once(
    pass2_path,
    "            imageio.mimsave(output_path, video_np, fps=12, codec='libx264',\n                           quality=8, pixelformat='yuv420p')\n\n            logger.info(f\"  ✓ Success: {output_path}\")",
    "            imageio.mimsave(output_path, video_np, fps=12, codec='libx264',\n                           quality=8, pixelformat='yuv420p')\n            if temporal_metrics_result is not None:\n                temporal_metrics_evaluator.write(temporal_metrics_result, str(output_path.with_suffix(\"\")))\n\n            logger.info(f\"  ✓ Success: {output_path}\")",
)

print("Patch aplicado com sucesso.")

Patch aplicado com sucesso.


In [5]:
%cd /content/void-model
!PYTHONPATH=/content/void-model python tests/test_temporal_metrics.py
!python -m py_compile videox_fun/utils/temporal_metrics.py inference/cogvideox_fun/predict_v2v.py inference/cogvideox_fun/inference_with_pass1_warped_noise.py tests/test_temporal_metrics.py

/content/void-model
2026-05-05 21:48:22.128 | INFO     | videox_fun.utils.temporal_metrics:write:249 - [temporal_eval] demo pass1 | pairs=2 | lpips=0.5000 | ofc_l1=0.2500 | psnr=12.04 | ssim=0.4008
..
----------------------------------------------------------------------
Ran 2 tests in 0.164s

OK


In [6]:
from pathlib import Path

REPO = Path('/content/void-model')
DATA_ROOTDIR = REPO / 'sample'
RUN_SEQS = 'lime'

BASE_MODEL_PATH = Path('/content/models/CogVideoX-Fun-V1.5-5b-InP')
PASS1_CHECKPOINT = Path('/content/models/void_pass1.safetensors')
PASS2_CHECKPOINT = Path('/content/models/void_pass2.safetensors')

PASS1_OUTPUT = REPO / 'outputs_pass1_temporal'
PASS2_OUTPUT = REPO / 'outputs_pass2_temporal'
WARPED_NOISE_CACHE = REPO / 'pass1_warped_noise_cache'

for path in [REPO / 'inference', BASE_MODEL_PATH, PASS1_CHECKPOINT, PASS2_CHECKPOINT]:
    print(path, 'OK' if path.exists() else 'FALTANDO')

/content/void-model/inference OK
/content/models/CogVideoX-Fun-V1.5-5b-InP OK
/content/models/void_pass1.safetensors OK
/content/models/void_pass2.safetensors OK


In [7]:
%cd /content/void-model
!python inference/cogvideox_fun/predict_v2v.py --help

/content/void-model
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.

       USAGE: inference/cogvideox_fun/predict_v2v.py [flags]
flags:

inference/cogvideox_fun/predict_v2v.py:
  --config: Path to the python config file
    (default: 'config/quadmask_cogvideox.py')

Try --helpfull to get a list of all flags.


In [12]:
%cd /content/void-model
!set -o pipefail; python inference/cogvideox_fun/predict_v2v.py \
  --config config/quadmask_cogvideox.py \
  --config.data.data_rootdir="{DATA_ROOTDIR}" \
  --config.experiment.run_seqs="{RUN_SEQS}" \
  --config.experiment.save_path="{PASS1_OUTPUT}" \
  --config.experiment.temporal_eval=True \
  --config.video_model.model_name="{BASE_MODEL_PATH}" \
  --config.video_model.transformer_path="{PASS1_CHECKPOINT}" \
  2>&1 | tee /content/pass1_full_log.txt ; echo "EXIT_CODE=$?"

/content/void-model
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/bin/bash: line 1: 10355 Killed                  python inference/cogvideox_fun/predict_v2v.py --config config/quadmask_cogvideox.py --config.data.data_rootdir="/content/void-model/sample" --config.experiment.run_seqs="lime" --config.experiment.save_path="/content/void-model/outputs_pass1_temporal" --config.experiment.temporal_eval=True --config.video_model.model_name="/content/models/CogVideoX-Fun-V1.5-5b-InP" --config.video_model.transformer_path="/content/models/void_pass1.safetensors" 2>&1
     10356 Done                    | tee /content/pass1_full_log.txt
EXIT_CODE=137


In [10]:
from pathlib import Path
out_dir = Path("/content/void-model/outputs_pass1_temporal")
print("Existe?", out_dir.exists())
if out_dir.exists():
    for f in sorted(out_dir.glob("*")):
        print(f.name)

Existe? False


In [13]:
!tail -n 120 /content/pass1_full_log.txt

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [14]:
!head -n 80 /content/pass1_full_log.txt

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [15]:
import torch
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0).total_memory / 1024**3, "GB")

Tesla T4
14.56317138671875 GB


In [16]:
import torch

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name)
    print("VRAM (GB):", round(props.total_memory / 1024**3, 2))

CUDA: True
GPU: Tesla T4
VRAM (GB): 14.56


In [ ]:
import json
from pathlib import Path

pass1_metrics = sorted(Path(PASS1_OUTPUT).glob('*_temporal_metrics.json'))
print('Arquivos JSON Pass 1:', len(pass1_metrics))

for path in pass1_metrics[:3]:
    print('\n', path.name)
    data = json.loads(path.read_text(encoding='utf-8'))
    print(json.dumps(data['summary'], indent=2))

In [ ]:
%cd /content/void-model
!python inference/cogvideox_fun/inference_with_pass1_warped_noise.py \
  --video_name "{RUN_SEQS}" \
  --data_rootdir "{DATA_ROOTDIR}" \
  --pass1_dir "{PASS1_OUTPUT}" \
  --output_dir "{PASS2_OUTPUT}" \
  --model_name "{BASE_MODEL_PATH}" \
  --model_checkpoint "{PASS2_CHECKPOINT}" \
  --warped_noise_cache_dir "{WARPED_NOISE_CACHE}" \
  --temporal_eval

In [ ]:
import json
from pathlib import Path

pass2_metrics = sorted(Path(PASS2_OUTPUT).glob('*_temporal_metrics.json'))
print('Arquivos JSON Pass 2:', len(pass2_metrics))

for path in pass2_metrics[:3]:
    print('\n', path.name)
    data = json.loads(path.read_text(encoding='utf-8'))
    print(json.dumps(data['summary'], indent=2))